# 📘 TÀI LIỆU KỸ THUẬT: MODULE M2 (ORIENTATION ALIGNMENT)

**Dự án:** Hệ thống AI Đọc chỉ số Đồng hồ nước (Water Meter Reading)  
**Phân hệ:** M2 - Căn chỉnh góc nghiêng  
**Người thực hiện:** Trần Xuân Đạt (Thực tập sinh)  
**Đơn vị:** Văn phòng Quản trị số  
**Ngày cập nhật:** 03/02/2026

---

## 1. Tổng quan (Executive Summary)

Trong quy trình đọc chỉ số đồng hồ nước tự động, ảnh đầu vào thường bị xoay ngẫu nhiên từ $0^\circ$ đến $360^\circ$, gây khó khăn cho việc nhận diện ký tự (OCR).

**Module M2** có nhiệm vụ:
1.  Tiếp nhận ảnh crop mặt đồng hồ.
2.  Dự đoán chính xác góc lệch bằng mạng Nơ-ron (ResNet18).
3.  Tự động xoay ảnh về trạng thái thẳng đứng (Upright) mà không làm mất chi tiết ảnh.

## 2. Cơ sở Lý thuyết & Toán học

### 2.1. Vấn đề của Hồi quy trực tiếp
Dự đoán trực tiếp số đo góc (Degree) gây ra điểm gián đoạn tại $0^\circ / 360^\circ$. Ví dụ: $1^\circ$ và $359^\circ$ rất gần nhau về hình ảnh nhưng xa nhau về giá trị số học, khiến hàm Loss khó hội tụ.

### 2.2. Giải pháp: Hồi quy trên Vòng tròn Đơn vị
Mô hình dự đoán vector hướng $\mathbf{v} = [P_{\sin}, P_{\cos}]$. Góc nghiêng $\theta$ (Radian) được khôi phục bằng hàm **Arctan 2 tham số**:

$$ \theta_{rad} = \operatorname{atan2}(P_{\sin}, P_{\cos}) $$

**Ưu điểm:** Hàm `atan2` phân biệt rõ 4 góc phần tư, giúp xác định chính xác hướng xoay $360^\circ$ mà không bị lỗi "lộn ngược đầu" ($180^\circ$) như hàm `atan` thông thường.

## 3. Giải thuật "Smart Rotate" (Xoay thông minh)

Khi xoay ảnh chữ nhật, 4 góc thường bị cắt mất (Clipping). Chúng ta áp dụng thuật toán tính toán lại khung hình (Bounding Box) mới:

$$ W_{new} = |H \cdot \sin(\theta)| + |W \cdot \cos(\theta)| $$
$$ H_{new} = |H \cdot \cos(\theta)| + |W \cdot \sin(\theta)| $$

Sau đó thực hiện dịch chuyển tâm (Translation) để đảm bảo toàn bộ đồng hồ nằm gọn trong khung hình mới.

## 4. Mã nguồn triển khai (Implementation)
Dưới đây là Class `WaterMeterAligner` đóng gói trọn vẹn quy trình.

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# --- HÀM XOAY THÔNG MINH ---
def smart_rotate(image, angle):
    h, w = image.shape[:2]
    if angle == 0: return image
    
    center = (w // 2, h // 2)
    # Lấy ma trận xoay
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    
    # Tính kích thước khung mới để không mất góc
    cos_val = np.abs(M[0, 0])
    sin_val = np.abs(M[0, 1])
    new_w = int((h * sin_val) + (w * cos_val))
    new_h = int((h * cos_val) + (w * sin_val))
    
    # Dịch chuyển tâm
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]
    
    return cv2.warpAffine(image, M, (new_w, new_h), 
                          flags=cv2.INTER_CUBIC, 
                          borderMode=cv2.BORDER_REPLICATE)

# --- CLASS CHÍNH ---
class WaterMeterAligner:
    def __init__(self, model_path, device='cpu'):
        self.device = torch.device(device)
        
        # 1. Định nghĩa lại Model M2 (ResNet18)
        self.model = models.resnet18(weights=None)
        self.model.fc = nn.Identity()
        self.model.fc = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 2)
        )
        
        # 2. Load Weights
        try:
            state = torch.load(model_path, map_location=self.device)
            self.model.load_state_dict(state)
            self.model.to(self.device).eval()
            print("✅ Đã load Model thành công!")
        except Exception as e:
            print(f"❌ Lỗi load model: {e}")
        
        # 3. Transform
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def align(self, cv_image):
        """
        Input: Ảnh OpenCV (BGR)
        Output: Ảnh đã xoay thẳng, Góc lệch
        """
        # Predict
        pil_img = Image.fromarray(cv2.cvtColor(cv_image, cv2.COLOR_BGR2RGB))
        tensor = self.transform(pil_img).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            out = self.model(tensor).cpu().numpy()[0]
            
        # Decode Angle
        sin, cos = out[0], out[1]
        angle_rad = np.arctan2(sin, cos)
        angle_deg = np.degrees(angle_rad)
        current_angle = (angle_deg + 360) % 360
        
        # Xoay ngược lại để thẳng (Correction)
        rotation_needed = -current_angle
        aligned_img = smart_rotate(cv_image, rotation_needed)
        
        return aligned_img, current_angle

## 5. Kết luận
Module M2 đã hoàn thành thử nghiệm với sai số góc trung bình (MAE) **~1.57 độ**. Hệ thống sẵn sàng tích hợp vào pipeline chính để phục vụ bước tiếp theo: Cắt vùng số (ROI Extraction).